<a href="https://colab.research.google.com/github/everestkuang/GarbageDetectionYoloV5/blob/main/Yolo11_AI_Garbage_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
from IPython.display import Image, display
# clone YOLOv11 repository
!pip install ultralytics

In [ ]:
!pip install roboflow

In [ ]:
# install dependencies as necessary
import torch

from IPython.display import Image, clear_output  # to display images

# clear_output()
print('Setup complete. Using torch %s %s' % (torch.__version__, torch.cuda.get_device_properties(0) if torch.cuda.is_available() else 'CPU'))

Setup complete. Using torch 2.6.0+cu124 CPU


In [ ]:
# this is the YAML file Roboflow wrote for us that we're loading into this notebook with our data
from roboflow import Roboflow
rf = Roboflow(api_key="raifJZV5ydW0w0mCsbV3")
project = rf.workspace("material-identification").project("garbage-classification-3")
version = project.version(2)
dataset = version.download("yolov11")

%cat {dataset.location}/data.yaml

loading Roboflow workspace...
loading Roboflow project...
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 6
names: ['BIODEGRADABLE', 'CARDBOARD', 'GLASS', 'METAL', 'PAPER', 'PLASTIC']

roboflow:
  workspace: material-identification
  project: garbage-classification-3
  version: 2
  license: CC BY 4.0
  url: https://universe.roboflow.com/material-identification/garbage-classification-3/dataset/2

#Define Model and Architecture

In [ ]:
# define number of classes based on YAML
import yaml
with open(dataset.location + "/data.yaml", 'r') as stream:
    num_classes = str(yaml.safe_load(stream)['nc'])

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

#Train Model

In [ ]:
from ultralytics import YOLO
import sys
sys.path
sys.path.append('/content/')

# Load a model
## Changes the model
model = YOLO("yolo11m.pt")

# Train the model
train_results = model.train(
    data=f"{dataset.location}/data.yaml",  # path to dataset YAML
    epochs=25,  # number of training epochs
    imgsz=640, # training image size
    batch = -1,
    cache = False,
    device="0",  # device to run on, i.e. device=0 or device=0,1,2,3 or device=cpu
)

# Evaluate model performance on the validation set
metrics = model.val()

print(metrics)
# Export the model to ONNX format
path = model.export(format="ncnn")  # return path to exported model


Ultralytics 8.3.168 🚀 Python-3.11.13 torch-2.6.0+cu124 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: None
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


#Evaluate Performance

In [ ]:
# Start tensorboard
# Launch after you have started training
# logs save in the folder "runs"
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
# first, display our ground truth data
print("GROUND TRUTH TRAINING DATA:")
Image(filename='./runs/detect/train4/val_batch0_labels.jpg', width=900)


In [ ]:
print("Predicted TRAINING DATA:")
Image(filename='./runs/detect/train4/val_batch0_pred.jpg', width=900)

In [ ]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

#Export all results to G Drive


In [ ]:
# Change All_Result_train.zip to different types of experiment run
!zip -r ./All_Result_train_yolo11m_25.zip "/content/runs"


In [ ]:
#Change the filename to whatever you had up above
%cp /content/All_Result_train_yolo11m_25.zip "/content/gdrive/My Drive/Research Project with Kushal - 2025/Training"